### Notebook 范围

### 目的
扫描 hindcast case 和 cleaned 产品是否存在，并建立总览矩阵。

### 科学问题
哪些年份、初始化月份、INT/NOCOUPL、扰动类型可用于 source diagnosis 或反馈对比？

### 方法
扫描 /mnt/soclim0/public_data/weiji/Hindcast，识别 partial_O3、EPflux_daily_ubar、U、Z3、FWD、AO/NAM。

### 输出
outputs/figures/00_overview 和 outputs/tables/00_overview。

### 解读
绿色/蓝色表示可用配置，0003 标记为特殊分支。

### 注意
这是产品级 inventory；单成员缺失会在后续 notebook 的 logs 中记录。


### 导入与公共路径

### 目的
加载 Extention_analysis 的公共函数、数据路径、输出目录和绘图参数。

### 科学问题
保证所有 notebook 使用同一套变量定义，避免 EPFlux、O3、U、Z300 的窗口和符号约定互相漂移。

### 方法
使用 hindcast_extension_utils.py 中的 DATA_ROOT、HINDCAST_ROOT、BWCN_ROOT、B2000_ROOT、FIG_DIR、TAB_DIR、CACHE_DIR 和 LOG_DIR。

### 输出
无图；只打印工作目录和 Hindcast 数据目录。

### 解读
如果路径打印正确，后续代码块会使用同一套 cleaned hindcast 产品。

### 注意
所有写入都限制在 code_cleaned/Hindcast_experiment/Extention_analysis/outputs 下；不会修改原始数据。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 绘制 availability matrix

### 目的
生成 case inventory 表和可用性矩阵。

### 科学问题
哪些 case 适合做多案例机制稳健性检验？哪些 case 适合 INT vs NOCOUPL？

### 方法
按 year-init_month 聚合 case，单元格写出 INT/NOCOUPL/v2/v3 和成员数。

### 输出
00_hindcast_availability_matrix.png/pdf 和 00_case_inventory.csv，放在 00_overview 子目录。

### 解读
同一年同月同时有 INT 与 NOCOUPL 时，可以用于反馈对比；只有 INT 时只能用于机制或路径检验。

### 注意
0003 是特殊分支，不强行配 NOCOUPL。


In [ ]:
fig_dir = figure_dir("00_overview")
tab_dir = table_dir("00_overview")
inv = discover_hindcast_cases()
inv_path = tab_dir / "00_case_inventory.csv"
inv.to_csv(inv_path, index=False)
# Keep a root-level copy for downstream synthesis compatibility.
inv.to_csv(TAB_DIR / "00_case_inventory.csv", index=False)
print(inv[["case", "year", "init_month", "config", "perturbation", "n_members", "can_source_diagnose"]].to_string(index=False))

if inv.empty:
    fig = empty_figure("No hindcast cases found", "Hindcast availability")
else:
    years = SELECTED_YEARS
    months = ["01", "02", "03", "04"]
    fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
    ax.set_xlim(-0.5, len(months) - 0.5)
    ax.set_ylim(len(years) - 0.5, -0.5)
    ax.set_xticks(range(len(months)), ["Jan", "Feb", "Mar", "Apr"])
    ax.set_yticks(range(len(years)), years)
    ax.set_title("Hindcast availability matrix")
    for yi, year in enumerate(years):
        for mi, month in enumerate(months):
            sub = inv[(inv["year"] == year) & (inv["init_month"] == month)]
            if sub.empty:
                color = "#f0f0f0"; text = "NA"
            else:
                has_pair = set(sub["config"]) >= {"INT", "NOCOUPL"}
                color = "#c7e9c0" if has_pair else ("#d9e8fb" if year != "0003" else "#fdd0a2")
                labels = []
                for _, row in sub.iterrows():
                    pert = {"small_temperature": "small", "v2_large_temperature": "v2", "v3_humidity": "v3"}.get(row["perturbation"], row["perturbation"])
                    labels.append(f"{row['config']} {pert}\nn={int(row['n_members'])}")
                text = "\n".join(labels)
            ax.add_patch(plt.Rectangle((mi - 0.5, yi - 0.5), 1, 1, facecolor=color, edgecolor="white", lw=2))
            ax.text(mi, yi, text, ha="center", va="center", fontsize=8)
    ax.grid(False)

savefig(fig, "00_hindcast_availability_matrix", fig_dir=fig_dir, notebook="00_case_inventory_and_utils.ipynb", scientific_question="哪些 hindcast cases 可用于 source diagnosis 和 INT/NOCOUPL 对比？", variables_windows="case inventory; partial_O3, EPflux_daily_ubar, U, Z3, FWD, AO/NAM", interpretation="同一年同月 INT+NOCOUPL 表示可做 feedback comparison；0003 为特殊分支。", caveat="这是产品级检查，不代表每个变量每个成员都完整。", csv_table=inv_path)
plt.show()
write_figure_guide()
